# Tech Challenge Fase 1 — Análise Exploratória de Dados (EDA)

## Objetivo
Este notebook realiza a **análise exploratória completa** do dataset Breast Cancer Wisconsin (Diagnostic),
com foco na identificação de padrões relacionados ao diagnóstico de câncer de mama (maligno vs benigno).

### Dataset
- **Fonte**: [Breast Cancer Wisconsin (Diagnostic) — UCI/Kaggle](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data)
- **Características**: 30 features numéricas extraídas de imagens digitalizadas de aspirados por agulha fina (FNA) de massas mamárias.
- **Target**: 0 = maligno, 1 = benigno
- **Amostras**: 569 (212 maligno, 357 benigno)

### Problema
Uma rede de hospitais busca implementar um sistema inteligente de suporte ao diagnóstico de câncer de mama,
auxiliando profissionais de saúde na identificação precoce de tumores malignos a partir de características
celulares extraídas de exames de FNA.

## 1. Carregamento e Inspeção Inicial

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Carregar dataset
data = load_breast_cancer(as_frame=True)
df = pd.concat([data.data, data.target.rename('target')], axis=1)
df['diagnosis'] = df['target'].map({0: 'maligno', 1: 'benigno'})

print(f'Shape do dataset: {df.shape}')
print(f'Número de features: {df.shape[1] - 2}')  # excluindo target e diagnosis
df.head()

In [ ]:
# Informações gerais do dataset
print('=== Tipos de dados ===')
print(df.dtypes)
print()
print('=== Valores ausentes ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Nenhum valor ausente encontrado.')
print()
print('=== Duplicatas ===')
print(f'Número de linhas duplicadas: {df.duplicated().sum()}')

### Observações
- O dataset não possui valores ausentes, o que simplifica o pré-processamento.
- Todas as 30 features são numéricas (float64), eliminando a necessidade de encoding categórico.
- Não há registros duplicados.

## 2. Estatísticas Descritivas

In [ ]:
# Estatísticas descritivas completas
desc = df.drop(columns=['diagnosis']).describe().T
desc['cv'] = desc['std'] / desc['mean']  # coeficiente de variação
desc.round(4)

In [ ]:
# Estatísticas por classe (maligno vs benigno)
features = data.data.columns.tolist()
stats_by_class = df.groupby('diagnosis')[features].agg(['mean', 'std']).T
print('=== Médias e desvios por diagnóstico (amostra das 10 primeiras features) ===')
stats_by_class.head(20)

### Observações
- Tumores malignos apresentam valores significativamente maiores em features como `mean radius`, `mean area`, `mean concavity` e `worst concave points`.
- O coeficiente de variação (CV) é elevado para features de "area" e "perimeter", indicando alta variabilidade — o que é esperado dado que tumores malignos tendem a ser maiores.
- As diferenças entre classes são mais acentuadas nas features do grupo "worst" (piores valores), sugerindo que extremos celulares são bons indicadores de malignidade.

## 3. Distribuição do Target (Classes)

In [ ]:
# Distribuição das classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contagem
class_counts = df['diagnosis'].value_counts()
axes[0].bar(class_counts.index, class_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição do Diagnóstico')
axes[0].set_ylabel('Contagem')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Percentual
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Proporção do Diagnóstico')

plt.tight_layout()
plt.savefig('../reports/eda_class_distribution.png', bbox_inches='tight')
plt.show()

print(f'Benigno: {class_counts["benigno"]} ({class_counts["benigno"]/len(df)*100:.1f}%)')
print(f'Maligno: {class_counts["maligno"]} ({class_counts["maligno"]/len(df)*100:.1f}%)')

### Observações
- O dataset apresenta leve desbalanceamento: ~63% benigno vs ~37% maligno.
- Esse desbalanceamento é moderado e não requer técnicas agressivas de balanceamento (como SMOTE), mas deve ser considerado na escolha de métricas — **recall** é mais importante que accuracy para evitar falsos negativos (não diagnosticar um tumor maligno).

## 4. Distribuição das Features

In [ ]:
# Histogramas das features "mean" por classe
mean_features = [f for f in features if f.startswith('mean')]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for i, feat in enumerate(mean_features):
    for diag, color in [('benigno', '#2ecc71'), ('maligno', '#e74c3c')]:
        subset = df[df['diagnosis'] == diag]
        axes[i].hist(subset[feat], bins=25, alpha=0.6, color=color, label=diag, density=True)
    axes[i].set_title(feat.replace('mean ', ''), fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribuição das Features "Mean" por Diagnóstico', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_mean_features_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# Box plots das features mais discriminativas
top_features = ['worst concave points', 'worst area', 'mean concave points',
                'worst radius', 'mean concavity', 'worst perimeter']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, feat in enumerate(top_features):
    sns.boxplot(data=df, x='diagnosis', y=feat, ax=axes[i],
                palette={'benigno': '#2ecc71', 'maligno': '#e74c3c'})
    axes[i].set_title(feat)

plt.suptitle('Box Plots das Features Mais Discriminativas', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_boxplots.png', bbox_inches='tight')
plt.show()

### Observações
- Features como `worst concave points`, `worst area` e `mean concave points` mostram separação clara entre classes, sugerindo alto poder discriminativo.
- Tumores malignos apresentam distribuições com valores mais altos e maior dispersão, especialmente em features de tamanho (area, radius, perimeter) e forma (concavity, concave points).
- Algumas features apresentam outliers em ambas as classes, indicando heterogeneidade tumoral.

## 5. Análise de Correlação

In [ ]:
# Heatmap de correlação completo
corr_matrix = df[features + ['target']].corr()

plt.figure(figsize=(18, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlação — Todas as Features', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Top correlações com o target
target_corr = corr_matrix['target'].drop('target').sort_values()

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.set_xlabel('Correlação com Target (Diagnóstico)')
ax.set_title('Correlação de Pearson de cada Feature com o Diagnóstico')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('../reports/eda_target_correlation.png', bbox_inches='tight')
plt.show()

print('=== Top 10 features mais correlacionadas (valor absoluto) ===')
print(target_corr.abs().sort_values(ascending=False).head(10))

In [ ]:
# Identificar pares de features altamente correlacionadas (potencial multicolinearidade)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlação': round(corr_matrix.iloc[i, j], 4)
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlação', ascending=False)
print(f'Pares de features com |correlação| > 0.9: {len(high_corr_df)}')
high_corr_df

### Observações sobre Correlação
- **Alta multicolinearidade** entre features de tamanho: `radius`, `perimeter` e `area` são altamente correlacionadas entre si (>0.95), como esperado geometricamente.
- Features com maior correlação negativa com o target (maligno=0): `worst concave points` (-0.79), `worst perimeter` (-0.78), `mean concave points` (-0.78) — indicando que valores altos dessas features estão associados a tumores malignos.
- A multicolinearidade sugere que modelos lineares (Logistic Regression) podem se beneficiar de regularização (já utilizamos na implementação com `max_iter`), enquanto modelos baseados em árvores (Random Forest) são naturalmente robustos a isso.
- Embora existam features redundantes, optamos por manter todas para preservar a informação completa e porque o Random Forest seleciona automaticamente as mais relevantes.

## 6. Padrões Específicos de Saúde Feminina

In [ ]:
# Scatter plot: área vs concavidade, colorido por diagnóstico
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for diag, color, marker in [('benigno', '#2ecc71', 'o'), ('maligno', '#e74c3c', 's')]:
    subset = df[df['diagnosis'] == diag]
    axes[0].scatter(subset['mean area'], subset['mean concavity'],
                   c=color, label=diag, alpha=0.6, marker=marker, s=30)
axes[0].set_xlabel('Mean Area')
axes[0].set_ylabel('Mean Concavity')
axes[0].set_title('Área Média vs Concavidade Média')
axes[0].legend()

for diag, color, marker in [('benigno', '#2ecc71', 'o'), ('maligno', '#e74c3c', 's')]:
    subset = df[df['diagnosis'] == diag]
    axes[1].scatter(subset['worst radius'], subset['worst concave points'],
                   c=color, label=diag, alpha=0.6, marker=marker, s=30)
axes[1].set_xlabel('Worst Radius')
axes[1].set_ylabel('Worst Concave Points')
axes[1].set_title('Pior Raio vs Piores Pontos Côncavos')
axes[1].legend()

plt.suptitle('Padrões Celulares — Indicadores de Malignidade', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_scatter_patterns.png', bbox_inches='tight')
plt.show()

### Padrões Identificados
- Tumores malignos tendem a apresentar **maior área celular** e **maior concavidade**, refletindo irregularidades no contorno celular típicas de células neoplásicas.
- A combinação de `worst radius` com `worst concave points` mostra separação visual clara entre as classes, sugerindo que essas features em conjunto são fortes preditores.
- Esses achados são consistentes com a literatura médica: células malignas tipicamente apresentam núcleos maiores, contornos irregulares e maior variabilidade morfológica.

## 7. Resumo da EDA

| Aspecto | Achado |
|---------|--------|
| **Qualidade dos dados** | Sem valores ausentes, sem duplicatas, todas as features numéricas |
| **Desbalanceamento** | Moderado (63% benigno vs 37% maligno) — recall é métrica prioritária |
| **Multicolinearidade** | Alta entre features de tamanho (radius/perimeter/area) |
| **Features discriminativas** | `worst concave points`, `worst area`, `mean concave points` |
| **Padrões clínicos** | Tumores malignos = maior área, maior concavidade, contornos irregulares |

### Implicações para Modelagem
- O **Standard Scaling** é necessário para Logistic Regression (sensível a escalas diferentes).
- Random Forest é robusto à multicolinearidade e pode capturar interações não-lineares.
- **Recall** deve ser a métrica principal: em diagnóstico de câncer, falsos negativos (não detectar um tumor maligno) são mais perigosos que falsos positivos.
- SHAP será utilizado para interpretar as previsões e garantir que o modelo está usando features clinicamente relevantes.